[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syriascitech/Medad-CV-Bootcamp/blob/main/Week7/week_7_training_and_finetuning.ipynb)


<style>
.rtl-cell {
    direction: rtl !important;
    text-align: right !important;
    line-height: 1.9 !important;
    font-size: 17px !important;
    font-family: Arial, Tahoma, sans-serif !important;
}
.rtl-cell p,
.rtl-cell h1, .rtl-cell h2, .rtl-cell h3, .rtl-cell h4,
.rtl-cell li, .rtl-cell blockquote, .rtl-cell th, .rtl-cell td {
    direction: rtl !important;
    text-align: right !important;
}
.rtl-cell ul, .rtl-cell ol {
    direction: rtl !important;
    text-align: right !important;
    padding-right: 2em !important;
    padding-left: 0 !important;
}
.rtl-cell blockquote {
    border-right: 4px solid #d0d7de !important;
    border-left: none !important;
    margin-right: 0 !important;
    padding-right: 1em !important;
}
.rtl-cell table {
    direction: rtl !important;
    text-align: right !important;
    margin-right: 0 !important;
    margin-left: auto !important;
    border-collapse: collapse !important;
}
.rtl-cell th, .rtl-cell td { padding: 7px 10px !important; }
.rtl-cell code { direction: ltr !important; unicode-bidi: isolate !important; }
.rtl-cell pre, .rtl-cell pre code {
    direction: ltr !important;
    text-align: left !important;
    unicode-bidi: embed !important;
}
.rtl-cell .todo {
    background-color: rgba(249, 168, 37, 0.14);
    border-right: 5px solid #F9A825;
    padding: 12px 18px;
    margin: 16px 0;
    border-radius: 6px;
}
.rtl-cell .note {
    background-color: rgba(127, 127, 127, 0.12);
    border-right: 5px solid #777;
    padding: 12px 18px;
    margin: 16px 0;
    border-radius: 6px;
}
</style>

<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h1>الأسبوع 7: التدريب والضبط الدقيق</h1>
<h2>Training and Fine-Tuning</h2>
<hr />
<p><strong>مسار أنظمة المرور الذكية – الرؤية الحاسوبية</strong></p>
<p><strong>المُعد/المؤلف: المهندس حسن صوان , المهندس عامر صوان</strong> </p>
<hr />

<p>في الأسبوع الماضي شغّلنا نموذج YOLO جاهزاً وكشفنا به السيارات والمشاة،
ولم ندرّب شيئاً. النموذج الجاهز يعرف ثمانين فئة تعلّمها من مجموعة
<strong>COCO</strong>، وكان ذلك كافياً لأن السيارة والشخص من ضمن هذه الفئات.</p>

<p>لكن نظام المرور الذي نبنيه يحتاج شيئاً لا تعرفه COCO إطلاقاً:</p>

<blockquote>
<p><strong>هل هذه المركبة سيارة إسعاف؟</strong></p>
</blockquote>

<p>لا توجد فئة اسمها <code>ambulance</code> بين فئات COCO الثمانين. لذلك مهما
ضبطنا قيمة <code>conf</code> ومهما غيّرنا حجم النموذج، لن يخبرنا النموذج
الجاهز بوجود إسعاف، لأنه ببساطة لا يملك الكلمة في قاموسه.</p>

<p>هنا يبدأ عمل هذا الأسبوع: سنأخذ النموذج الجاهز، ونُعلّمه فئة جديدة من
بياناتنا نحن. هذه العملية اسمها <strong>Fine-Tuning</strong>، وهي الجسر بين
"نموذج يعمل" و"نموذج يعمل على مشكلتنا".</p>

<p>وفي نهاية الدرس سيكون بين أيدينا نموذج يميّز الإسعاف عن بقية المركبات،
وهو النموذج الذي سنتتبّعه في الأسبوع القادم ونبني عليه منطق أولوية الإشارة
في الأسبوع التاسع.</p>

<div class="note">
<p><strong>هذا الدرس يعمل بلا إنترنت.</strong> كل ما تحتاجه موجود داخل مجلد
<code>Week7</code> نفسه: الصور، والتوسيم، والنموذج الجاهز، ونتائج التدريب
المرجعية. لا تحتاج إلى حساب ولا إلى تحميل أي شيء أثناء الحصة.</p>
</div>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>أهداف الدرس</h2>

<p>في نهاية هذا الدرس ستكون قادراً على أن:</p>

<ol>
<li>تشرح لماذا يفشل نموذج جاهز على مسألة معيّنة، وتميّز بين نوعَي الفشل:
<strong>فجوة في الفئات</strong> و<strong>اختلاف في المجال</strong>.</li>
<li>تفرّق بين التدريب من الصفر و<strong>Transfer Learning</strong>، وتعرف
متى يستحق كلٌّ منهما.</li>
<li>تقرأ وتكتب توسيماً بصيغة YOLO، وتحوّل إحداثيات البكسل إلى إحداثيات
مطبَّعة بيدك.</li>
<li>تبني بنية مجلدات صحيحة وملف <code>data.yaml</code>، وتقسّم البيانات إلى
<code>train</code> و <code>valid</code> و <code>test</code> تقسيماً نزيهاً.</li>
<li>تشغّل عملية تدريب فعلية وتفهم معنى كل رقم يظهر أثناءها.</li>
<li>تقيّم النموذج بـ <strong>Precision</strong> و<strong>Recall</strong>
و<strong>mAP</strong>، وتقرأ مصفوفة الالتباس ومنحنيات التدريب.</li>
<li>تشخّص التجهيز الزائد <strong>Overfitting</strong> وتعرف ماذا تفعل حياله.</li>
</ol>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>محتويات الدرس</h2>

<ol>
<li>أين يفشل النموذج الجاهز؟</li>
<li>ما هو Fine-Tuning ولماذا لا ندرّب من الصفر؟</li>
<li>تجهيز قاعدة البيانات</li>
<li>التعزيز Data Augmentation</li>
<li>تشريح عملية التدريب</li>
<li>لنُدرّب</li>
<li>قراءة النتائج والتقييم</li>
<li>اللحظة الحاسمة: قبل وبعد</li>
<li>دليل التحسين حين تكون النتائج ضعيفة</li>
<li>حفظ النموذج واستخدامه لاحقاً</li>
<li>حدود ما فعلناه</li>
<li>تمرين صفي</li>
<li>أسئلة مراجعة سريعة</li>
<li>بنك أسئلة Kahoot</li>
<li>الخلاصة</li>
</ol>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>التهيئة</h1>

</div>


In [ ]:
import sys


!{sys.executable} -m pip install -q ultralytics opencv-python pandas matplotlib pyyaml numpy


!{sys.executable} -m pip install -q torch torchvision

In [ ]:
import importlib.util

REQUIRED = ["ultralytics", "cv2", "torch", "pandas", "matplotlib", "yaml"]

missing = [name for name in REQUIRED if importlib.util.find_spec(name) is None]

if missing:
    print("المكتبات الناقصة:", "، ".join(missing))
    print("نفّذ ما في ملف SETUP.md مرة واحدة، ثم أعد تشغيل النواة (Restart Kernel).")
else:
    print("كل المكتبات المطلوبة متوفرة.")


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>الاستيرادات وإعدادات العمل بلا إنترنت</h2>

<p>مكتبة Ultralytics تحاول الاتصال بالإنترنت في أكثر من موضع: لتحميل الأوزان
إن لم تجدها، ولإرسال إحصاءات استخدام. الخلية التالية تعطّل ذلك وتشير إلى
نسخة النموذج المحفوظة محلياً داخل <code>models/</code>.</p>

</div>


In [ ]:
import os
from collections import Counter
from pathlib import Path

os.environ["YOLO_VERBOSE"] = "False"

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml



os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import matplotlib.pyplot as plt
import cv2

from ultralytics import YOLO, settings

# إيقاف إرسال إحصاءات الاستخدام، فهو يحتاج اتصالاً بالشبكة.
settings.update({"sync": False})

# مسارات محلية: لا شيء يُحمَّل من الإنترنت في هذا الدفتر.
PRETRAINED_PATH = "models/yolo11n.pt"
DATA_YAML = "dataset/data.yaml"
REFERENCE_RUN = Path("runs_reference/emergency_full")

print("PyTorch:", torch.__version__)
print("النموذج الجاهز:", PRETRAINED_PATH)


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>الجهاز المستخدم</h2>

<p>سنعمل في الحصة على المعالج <code>cpu</code>، وهذا يحدّد حجم التدريب الذي
يمكننا تنفيذه مباشرة. إن كان لديك كرت شاشة NVIDIA في البيت فسيظهر
<code>cuda</code>، وعندها يمكنك تشغيل التدريب الكامل في القسم السادس.</p>

</div>


In [ ]:
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("الجهاز المستخدم:", DEVICE)
print("عدد أنوية المعالج المتاحة:", os.cpu_count())


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>1. أين يفشل النموذج الجاهز؟</h1>

<p>قبل أن ندرّب أي شيء، علينا أن نثبت أن التدريب ضروري أصلاً. أسوأ ما يمكن
أن يفعله مهندس رؤية حاسوبية هو أن يدرّب نموذجاً جديداً بينما كان النموذج
الجاهز كافياً.</p>

<p>لذلك سنبدأ بتجربة: نأخذ النموذج نفسه الذي استخدمناه في الأسبوع الماضي،
ونشغّله على صور فيها سيارات إسعاف، ثم ننظر ماذا يقول.</p>

<h2>1.1 تجربة حية على صور من الشارع</h2>

<p>مجلد <code>failures/</code> يحوي خمس صور اخترناها بعناية، وملف
<code>failures.csv</code> يذكر لكل صورة ما <em>ينبغي</em> أن يكون الجواب
الصحيح.</p>

</div>


In [ ]:
pretrained_model = YOLO(PRETRAINED_PATH)

print("عدد الفئات التي يعرفها النموذج الجاهز:", len(pretrained_model.names))

# هل توجد فئة للإسعاف بين فئات COCO الثمانين؟
has_ambulance = any(
    "ambulance" in name.lower() for name in pretrained_model.names.values()
)
print("هل يعرف النموذج فئة ambulance؟", "نعم" if has_ambulance else "لا")

# الفئات المتعلقة بالمركبات التي يعرفها فعلاً:
vehicle_names = ["car", "bus", "truck", "motorcycle", "bicycle", "train"]
for class_id, name in pretrained_model.names.items():
    if name in vehicle_names:
        print(f"  {class_id:2d} -> {name}")


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>الآن نشغّل النموذج على صور الفشل ونقارن ما يقوله بما ينبغي أن يقوله.</p>

</div>


In [ ]:
failures_df = pd.read_csv("failures/failures.csv")

rows = []

for _, item in failures_df.iterrows():
    image_path = f"failures/{item['file']}"

    result = pretrained_model.predict(
        source=image_path,
        conf=0.25,
        device=DEVICE,
        verbose=False,
    )[0]

    if len(result.boxes) == 0:
        predicted = "لا شيء"
        confidence = 0.0
    else:
        # نأخذ أعلى كشف ثقةً كإجابة النموذج عن "ما هذه المركبة؟"
        best = int(result.boxes.conf.argmax())
        predicted = result.names[int(result.boxes.cls[best])]
        confidence = float(result.boxes.conf[best])

    rows.append({
        "الصورة": item["file"],
        "قال النموذج": predicted,
        "الثقة": round(confidence, 2),
        "الصحيح": item["correct_class"],
        "نوع الفشل": item["failure_type"],
    })

pd.DataFrame(rows)


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>ولنرَ ذلك بأعيننا لا بالأرقام فقط:</p>

</div>


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

for ax, (_, item) in zip(axes.flat, failures_df.iterrows()):
    result = pretrained_model.predict(
        source=f"failures/{item['file']}",
        conf=0.25,
        device=DEVICE,
        verbose=False,
    )[0]

    ax.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(f"should be: {item['correct_class']}", fontsize=13)
    ax.axis("off")

# الخلية السادسة نتركها فارغة لأن عدد الصور خمس.
axes.flat[-1].axis("off")

plt.suptitle("Pretrained YOLO on emergency vehicles", fontsize=17)
plt.tight_layout()
plt.show()


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>1.2 أنواع الفشل الأربعة</h2>

<p>ما رأيناه ليس فشلاً واحداً بل أربعة أنواع مختلفة، ولكل نوع علاج مختلف:</p>

<table>
<thead>
<tr><th>النوع</th><th>ما يحدث</th><th>مثال</th><th>هل يصلحه ضبط conf؟</th></tr>
</thead>
<tbody>
<tr>
  <td><strong>فجوة في الفئات</strong></td>
  <td>الفئة غير موجودة في قاموس النموذج أصلاً</td>
  <td>إسعاف يُكشف <code>truck</code></td>
  <td>لا، أبداً</td>
</tr>
<tr>
  <td><strong>فئة خاطئة</strong></td>
  <td>يكشف الجسم لكن يسمّيه خطأ</td>
  <td>ميكروباص يتذبذب بين <code>bus</code> و <code>truck</code></td>
  <td>لا</td>
</tr>
<tr>
  <td><strong>كشف مفقود</strong></td>
  <td>لا يرى الجسم إطلاقاً</td>
  <td>إسعاف ليلاً أو بعيداً</td>
  <td>جزئياً، بخفض العتبة</td>
</tr>
<tr>
  <td><strong>صندوق سيّئ</strong></td>
  <td>يرى الجسم لكن الصندوق غير دقيق</td>
  <td>مركبتان متلاصقتان في صندوق واحد</td>
  <td>لا</td>
</tr>
</tbody>
</table>

<p>النوعان الأول والثاني سببهما <strong>فجوة في الفئات</strong>: القاموس
نفسه ناقص. والنوعان الثالث والرابع سببهما <strong>اختلاف المجال
Domain Shift</strong>: الفئة موجودة لكن الصور التي تدرّب عليها النموذج تختلف
عن صورنا في الإضاءة والزاوية والازدحام.</p>

<blockquote>
<p>هذا التمييز هو مفتاح الدرس كله. <strong>فجوة الفئات لا يصلحها إلا
التدريب.</strong> أما اختلاف المجال فقد يخفّ بضبط العتبات، لكن التدريب على
بياناتنا يعالجه أفضل بكثير.</p>
</blockquote>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h4>1.3 لماذا يحدث هذا؟ ما هي COCO</h4>

<p>النموذج الجاهز تدرّب على مجموعة COCO، وهي ثمانون فئة عامة اختيرت قبل سنوات لأغراض بحثية عامة، لا لنظام مرور. الشكل التالي يوضح الفجوة بين ما تعرفه COCO وما نحتاجه نحن.</p>

<div style="text-align:center; margin:24px 0;">
<img
src="media/coco_class_gap.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week7/media/coco_class_gap.jpg';"
alt="Figure 1 - COCO class gap vs traffic system needs"
width="900">
</div>

</div>

<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>فكر قبل المتابعة</h2>

<p>قبل أن تنتقل إلى القسم التالي، فكّر في هذه الأسئلة:</p>

<ol>
<li>لو خفّضنا <code>conf</code> إلى <code>0.05</code>، هل سيظهر
<code>ambulance</code> في النتائج؟ ولماذا؟</li>
<li>النموذج سمّى الإسعاف <code>truck</code>. هل هذا خطأ من النموذج، أم أنه
أفضل إجابة ممكنة ضمن قاموسه؟</li>
<li>لو أردت نظاماً يعطي الأولوية للإسعاف، ما الحد الأدنى من الفئات التي
تحتاجها فعلاً؟</li>
<li>متى يكون استخدام النموذج الجاهز هو القرار الصحيح رغم أخطائه؟</li>
</ol>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>2. ما هو Fine-Tuning ولماذا لا ندرّب من الصفر؟</h1>

<h2>2.1 التدريب من الصفر مقابل Transfer Learning</h2>

<p>أمامنا طريقان لبناء نموذج يعرف الإسعاف:</p>

<table>
<thead>
<tr>
  <th></th>
  <th>التدريب من الصفر</th>
  <th>Fine-Tuning</th>
</tr>
</thead>
<tbody>
<tr>
  <td>نقطة البداية</td>
  <td>أوزان عشوائية</td>
  <td>أوزان نموذج تدرّب على ملايين الصور</td>
</tr>
<tr>
  <td>حجم البيانات المطلوب</td>
  <td>عشرات أو مئات الآلاف من الصور</td>
  <td>مئات إلى بضعة آلاف</td>
</tr>
<tr>
  <td>الزمن</td>
  <td>أيام إلى أسابيع</td>
  <td>دقائق إلى ساعات</td>
</tr>
<tr>
  <td>العتاد</td>
  <td>عدة كروت شاشة</td>
  <td>كرت واحد، وأحياناً معالج فقط</td>
</tr>
<tr>
  <td>متى نختاره</td>
  <td>بيانات مختلفة جذرياً عن الصور الطبيعية</td>
  <td>في كل الحالات العملية تقريباً</td>
</tr>
</tbody>
</table>

<p>في الممارسة العملية، التدريب من الصفر قرار نادر جداً. نحن سنستخدم
Fine-Tuning، وهذا ليس حلاً وسطاً بل هو الخيار الصحيح.</p>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h4>2.2 ماذا تعلّمه الـ Backbone فعلاً؟</h4>

<p>لفهم لماذا ينجح Fine-Tuning، علينا أن نعرف أن النموذج ليس كتلة واحدة. الطبقات الأولى تعلّمت أشياء عامة جداً: الحواف، ثم الملامس، ثم أجزاء الأجسام كالعجلات والنوافذ. هذه المعرفة صالحة لأي مركبة، ولا داعي لإعادة تعلّمها. الجزء الوحيد المرتبط بقائمة الفئات هو الرأس Head.</p>



<div style="text-align:center; margin:24px 0;">
<img
src="media/transfer_learning.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week7/media/transfer_learning.jpg';"
alt="Figure 2 - Transfer learning and the backbone"
width="900">
</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>2.3 متى يكون النموذج الجاهز كافياً ومتى نحتاج التدريب؟</h2>

<p>استخدم هذه القاعدة العملية:</p>

<ul>
<li><strong>الفئة التي تحتاجها غير موجودة في قاموس النموذج</strong>
  ← التدريب إلزامي. لا بديل عنه.</li>
<li><strong>الفئة موجودة لكن الأداء ضعيف على صورك</strong>
  ← جرّب أولاً ضبط <code>conf</code> و <code>imgsz</code>، ثم جرّب نموذجاً
  أكبر (<code>yolo11s</code> بدل <code>yolo11n</code>). فإن بقي الأداء ضعيفاً
  فالتدريب هو الحل.</li>
<li><strong>الفئة موجودة والأداء جيد</strong>
  ← لا تدرّب. استخدم الجاهز ووفّر وقتك لبقية النظام.</li>
</ul>

<p>حالتنا من النوع الأول: <code>ambulance</code> غير موجودة، فالقرار محسوم.</p>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>3. تجهيز قاعدة البيانات</h1>

<p>هذا أطول أقسام الدرس وأكثرها أهمية عملياً. في المشاريع الحقيقية يذهب
معظم الوقت إلى البيانات لا إلى النموذج، والنموذج الممتاز على بيانات سيّئة
يعطي نتائج سيّئة دائماً.</p>

<h2>3.1 تصنيف كامل الصورة، أم البحث عن جسم في هذه الصورة؟</h2>

<p>قبل أن نبدأ، لا بد من تمييز أساسي. ليست كل مجموعة بيانات فيها صور
ومركبات صالحةً لتدريب كاشف.</p>

<p>في مجلد <code>classification_example/</code> نسخة من ملف توسيم مجموعة
شهيرة لمركبات الطوارئ. لننظر ماذا تحوي:</p>

</div>


In [ ]:
classification_labels = pd.read_csv("classification_example/train.csv")

print("الأعمدة:", list(classification_labels.columns))
print("عدد الصفوف:", len(classification_labels))
print()
print(classification_labels.head())
print()
print("توزيع التسميات:")
print(classification_labels["emergency_or_not"].value_counts())


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>عمودان فقط: اسم الصورة، وهل فيها مركبة طوارئ أم لا. هذه تسمية
<strong>على مستوى الصورة</strong> (Image-level)، وهي تكفي لمسألة
<strong>تصنيف</strong> تجيب عن سؤال: "ماذا في هذه الصورة؟"</p>

<p>لكن الكشف يحتاج جواباً عن سؤال أصعب: "<strong>أين</strong> كل جسم،
و<strong>ما</strong> هو؟" وهذا يتطلب تسمية <strong>على مستوى الجسم</strong>
(Object-level): أربعة أرقام تحدّد صندوقاً، وفئة لكل صندوق، لكل جسم في
الصورة على حدة.</p>

<div class="note">
<p><strong>القاعدة:</strong> لا يمكن تدريب كاشف على مجموعة تصنيف. لو رأيت
مجموعة بيانات فيها ملف CSV بعمودين فقط، فهي للتصنيف لا للكشف مهما كان
اسمها. تحقق دائماً من وجود صناديق الإحاطة قبل أن تخطط لتدريب كاشف.</p>
</div>

<h2>3.2 من أين نجمع الصور؟</h2>

<p>القاعدة الذهبية في جمع بيانات الكشف:</p>

<blockquote>
<p><strong>التنوّع أهم من الكم.</strong></p>
</blockquote>

<p>مئتا صورة متنوعة تتفوق على ألفَي صورة متشابهة. تنوَّع في:</p>

<ul>
<li><strong>الإضاءة</strong>: نهار، ليل، غروب، أضواء شوارع.</li>
<li><strong>الطقس</strong>: صحو، مطر، غبار.</li>
<li><strong>الزاوية والبعد</strong>: من الأمام، من الجانب، من الخلف،
قريب وبعيد.</li>
<li><strong>الحجب</strong>: مركبات محجوبة جزئياً خلف غيرها.</li>
<li><strong>الخلفية</strong>: شارع مزدحم، طريق فارغ، ساحة مستشفى.</li>
<li><strong>الأمثلة السلبية</strong>: صور فيها مركبات عادية فقط، بلا إسعاف.
هذه ضرورية ليتعلّم النموذج ما <em>ليس</em> إسعافاً.</li>
</ul>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h4>3.3 صيغة YOLO للتوسيم</h4>

<p>لكل صورة ملف نصي واحد يحمل الاسم نفسه بامتداد <code>.txt</code>. كل سطر في الملف يمثّل جسماً واحداً، وفيه خمسة أرقام. 

الشكل التالي يشرح هذه الأرقام الخمسة.</p>
</div>

<div style="text-align:center; margin:24px 0;">
<img
src="media/yolo_label_format.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week7/media/yolo_label_format.jpg';"
alt="Figure 3 - YOLO label format"
width="900">
</div>

<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>3.4 نكتب ملف توسيم بأيدينا</h2>

<p>أفضل طريقة لفهم الصيغة هي أن نكتبها بأنفسنا مرة واحدة، بلا أي أداة
وبلا إنترنت. سنأخذ صورة من قاعدة بياناتنا، ونحوّل صندوقاً بالبكسل إلى
سطر YOLO، ثم نقرأ السطر ونعيد رسم الصندوق للتحقق.</p>

</div>


In [ ]:
# نختار صورة فيها إسعاف واحد لنعمل عليها
for candidate in sorted(Path("dataset/train/images").glob("*.jpg")):
    candidate_label = Path("dataset/train/labels") / (candidate.stem + ".txt")
    ambulance_lines = [
        line for line in candidate_label.read_text().split("\n")
        if line.strip() and line.split()[0] == "0"
    ]
    if len(ambulance_lines) == 1:
        sample_image_path = candidate
        break

sample_image = cv2.cvtColor(cv2.imread(str(sample_image_path)), cv2.COLOR_BGR2RGB)
image_height, image_width = sample_image.shape[:2]
print("الصورة:", sample_image_path.name, "->", image_width, "x", image_height)

# تخيّل أنك فتحت الصورة في أداة توسيم وقرأت زوايا الصندوق بالبكسل.
# هذه هي الأرقام التي كنت ستراها على الشاشة:
_, gt_xc, gt_yc, gt_w, gt_h = map(float, ambulance_lines[0].split())
x1 = round((gt_xc - gt_w / 2) * image_width)
y1 = round((gt_yc - gt_h / 2) * image_height)
x2 = round((gt_xc + gt_w / 2) * image_width)
y2 = round((gt_yc + gt_h / 2) * image_height)

print(f"الصندوق كما قرأناه بالبكسل: ({x1}, {y1}) -> ({x2}, {y2})")

# الخطوة 1: من الزوايا إلى المركز والأبعاد
box_width = x2 - x1
box_height = y2 - y1
x_center = x1 + box_width / 2
y_center = y1 + box_height / 2

print(f"بالبكسل  -> x_center={x_center:.0f}, y_center={y_center:.0f}, "
      f"w={box_width}, h={box_height}")

# الخطوة 2: التطبيع، أي القسمة على أبعاد الصورة
x_norm = x_center / image_width
y_norm = y_center / image_height
w_norm = box_width / image_width
h_norm = box_height / image_height

CLASS_ID = 0  # ambulance

label_line = f"{CLASS_ID} {x_norm:.6f} {y_norm:.6f} {w_norm:.6f} {h_norm:.6f}"
print("سطر YOLO ->", label_line)


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>الآن نكتب السطر إلى ملف، ثم نقرأه من جديد ونعيد بناء الصندوق
بالبكسل. إن عاد الصندوق إلى مكانه الأصلي فقد فهمنا الصيغة فهماً صحيحاً.</p>

</div>


In [ ]:
Path("demo_label.txt").write_text(label_line + "\n")

# القراءة من الملف كما ستقرأه YOLO تماماً
read_class_id, rx, ry, rw, rh = Path("demo_label.txt").read_text().split()
rx, ry, rw, rh = float(rx), float(ry), float(rw), float(rh)

# عكس التطبيع: من [0,1] إلى بكسل
back_x1 = round((rx - rw / 2) * image_width)
back_y1 = round((ry - rh / 2) * image_height)
back_x2 = round((rx + rw / 2) * image_width)
back_y2 = round((ry + rh / 2) * image_height)

print("الصندوق الأصلي   :", (x1, y1, x2, y2))
print("الصندوق بعد العودة:", (back_x1, back_y1, back_x2, back_y2))

drawn = sample_image.copy()
cv2.rectangle(drawn, (back_x1, back_y1), (back_x2, back_y2), (46, 125, 50), 3)

plt.figure(figsize=(8, 6))
plt.imshow(drawn)
plt.axis("off")
plt.title("Box reconstructed from the normalized label file")
plt.show()


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>3.5 أدوات التوسيم المستخدمة عملياً</h2>

<p>لن نوسّم قاعدة بيانات كاملة بأيدينا، فهذا عمل أسابيع. الأدوات التالية
هي ما يُستخدم فعلياً في المشاريع:</p>

<table>
<thead>
<tr>
  <th>الأداة</th>
  <th>في المتصفح</th>
  <th>تحتاج تسجيلاً</th>
  <th>تعمل بلا إنترنت</th>
  <th>متى نختارها</th>
</tr>
</thead>
<tbody>
<tr>
  <td><strong>LabelImg</strong></td>
  <td>لا</td><td>لا</td><td>نعم</td>
  <td>الأنسب لحالتنا: برنامج بسيط يعمل بلا إنترنت ويصدّر صيغة YOLO مباشرة</td>
</tr>
<tr>
  <td><strong>labelme</strong></td>
  <td>لا</td><td>لا</td><td>نعم</td>
  <td>حين نحتاج أشكالاً حرة (polygons) لا صناديق فقط</td>
</tr>
<tr>
  <td><strong>makesense.ai</strong></td>
  <td>نعم</td><td>لا</td><td>لا</td>
  <td>حين لا تملك صلاحية تثبيت برامج على الجهاز</td>
</tr>
<tr>
  <td><strong>Roboflow</strong></td>
  <td>نعم</td><td>نعم</td><td>لا</td>
  <td>المعياري صناعياً: توسيم وتعزيز وتصدير وإدارة نسخ</td>
</tr>
<tr>
  <td><strong>CVAT</strong></td>
  <td>نعم</td><td>نعم</td><td>نعم عند التنصيب محلياً</td>
  <td>الفرق الكبيرة وتوسيم الفيديو إطاراً إطاراً</td>
</tr>
</tbody>
</table>

<p>روابط للاطلاع خارج الحصة:
<a href="https://github.com/HumanSignal/labelImg">LabelImg</a> ·
<a href="https://www.makesense.ai/">makesense.ai</a> ·
<a href="https://roboflow.com/annotate">Roboflow Annotate</a> ·
<a href="https://www.cvat.ai/">CVAT</a></p>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h4>دورة توسيم كاملة في LabelImg</h4>

<p>اخترنا LabelImg لأنها الأداة الوحيدة في الجدول التي تعمل بلا إنترنت تماماً مثل بقية هذا الدرس. الشكل التالي يعرض الخطوات الأربع كاملة حتى لو لم تكن الأداة مثبّتة على جهازك.</p>

</div>

<div style="text-align:center; margin:24px 0;">
<img
src="media/annot_labelimg.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week7/media/annot_labelimg.jpg';"
alt="Figure 4 - Annotation workflow in LabelImg"
width="900">
</div>



<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>3.6 بنية المجلدات وملف data.yaml</h2>

<p>مكتبة Ultralytics تتوقع بنية محددة. لنفحص بنية قاعدة بياناتنا:</p>

</div>


In [ ]:
dataset_root = Path("dataset")

for split in ["train", "valid", "test"]:
    n_images = len(list((dataset_root / split / "images").glob("*.jpg")))
    n_labels = len(list((dataset_root / split / "labels").glob("*.txt")))
    print(f"{split:6s} -> {n_images:4d} صورة، {n_labels:4d} ملف توسيم")

print()
print("محتوى data.yaml:")
print("-" * 40)
print(Path("dataset/data.yaml").read_text())


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>لاحظ ثلاث نقاط مهمة:</p>

<ul>
<li>كل صورة <code>img_001.jpg</code> يقابلها ملف <code>img_001.txt</code>
بالاسم نفسه تماماً، في مجلد <code>labels</code> الموازي لمجلد
<code>images</code>. هذه ليست عادة تنظيمية بل هي الطريقة التي تعثر بها
Ultralytics على التوسيم.</li>
<li>ترتيب الأسماء في <code>names</code> هو الذي يحدّد
<code>class_id</code>. لو غيّرت الترتيب بعد التوسيم فسدت كل الملفات
دفعة واحدة.</li>
<li>لا يوجد مفتاح <code>path</code> في ملفنا، وهذا مقصود. حين يغيب
<code>path</code> تعتبر Ultralytics <strong>مجلد ملف data.yaml نفسه</strong>
جذراً للمسارات.</li>
</ul>

<div class="note">
<p><strong>فخّ يقع فيه كثيرون:</strong> لو كتبت <code>path: .</code> ظاناً
أنها تعني «المجلد الحالي لهذا الملف»، فستفاجأ بأن النقطة تُحلّ إلى
<strong>مجلد العمل الذي شُغّل منه الدفتر</strong> لا إلى مجلد قاعدة
البيانات. عندها يبحث التدريب عن <code>valid/images</code> في المكان
الخطأ ويفشل برسالة <code>images not found</code>.</p>
<p>الحل: إما أن تحذف <code>path</code> تماماً كما فعلنا، أو أن تكتب مساراً
مطلقاً كاملاً.</p>
</div>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h4>بنية المجلدات مقابل ملف data.yaml</h4>

<p>الشكل التالي يربط بين ما نراه على القرص وما نكتبه في ملف الإعداد، وهو أكثر موضع يقع فيه الطلاب في الخطأ عند أول تدريب لهم.</p>

</div>

<div style="text-align:center; margin:24px 0;">
<img
src="media/dataset_structure.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week7/media/dataset_structure.jpg';"
alt="Dataset folder structure vs data.yaml"
width="900">
</div>

<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>3.7 التقسيم train / valid / test</h2>

<p>نقسّم البيانات ثلاثة أقسام، ولكل قسم دور مختلف تماماً:</p>

<ul>
<li><strong>train</strong> (70%): النموذج يراها ويتعلّم منها ويعدّل أوزانه
بناءً عليها.</li>
<li><strong>valid</strong> (20%): النموذج لا يتعلّم منها، لكننا نقيس عليها
بعد كل epoch لنختار أفضل نسخة ونعرف متى نتوقف.</li>
<li><strong>test</strong> (10%): لا تُلمس إلا مرة واحدة في النهاية، لتقدير
نزيه للأداء الحقيقي.</li>
</ul>

<div class="note">
<p><strong>الخطأ الذي يفسد كل شيء:</strong> أن تضع صوراً شديدة التشابه في
<code>train</code> و <code>valid</code> معاً. لو أخذت إطارات متتالية من
فيديو واحد ووزّعتها عشوائياً، فسيرى النموذج في <code>valid</code> صوراً
تكاد تطابق ما حفظه من <code>train</code>، وستحصل على أرقام ممتازة كاذبة.
القاعدة: <strong>قسّم حسب المشهد لا حسب الصورة</strong>.</p>
</div>

<p>في قاعدة بياناتنا كل صورة لقطة مستقلة من مصوّر مختلف، فلا يوجد هذا
الخطر، والتقسيم العشوائي سليم هنا.</p>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h4>دور كل قسم من الأقسام الثلاثة</h4>

<p>الشكل التالي يلخّص الأدوار الثلاثة والخطأ الشائع في التقسيم.</p>

</div>


<div style="text-align:center; margin:24px 0;">
<img
src="media/train_val_test_split.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week7/media/train_val_test_split.jpg';"
alt="Figure 6 - Train / valid / test split roles"
width="900">
</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>3.8 أخطاء التوسيم الشائعة</h2>

<table>
<thead>
<tr><th>الخطأ</th><th>أثره على النموذج</th></tr>
</thead>
<tbody>
<tr><td>صندوق أوسع من الجسم بكثير</td>
    <td>يتعلّم النموذج أن الخلفية جزء من الجسم</td></tr>
<tr><td>صندوق أضيق يقصّ أطراف الجسم</td>
    <td>يتعلّم أجساماً ناقصة ويفشل على الكاملة</td></tr>
<tr><td>جسم موجود لكنه غير موسوم</td>
    <td>الأسوأ على الإطلاق: نعاقب النموذج حين يكون محقاً</td></tr>
<tr><td>الجسم نفسه موسوم بفئتين</td>
    <td>إشارتان متناقضتان على البكسلات نفسها</td></tr>
<tr><td>صندوق واحد يغطي مجموعة أجسام</td>
    <td>يتعلّم النموذج أن الكتلة جسم واحد</td></tr>
<tr><td>توسيم رسم أو صورة داخل صورة</td>
    <td>يتعلّم أن اللوحات والإعلانات مركبات حقيقية</td></tr>
<tr><td>تسمية غير متسقة بين الموسِّمين</td>
    <td>حدود الفئات تصبح ضبابية</td></tr>
<tr><td>تجاهل الأجسام المحجوبة جزئياً</td>
    <td>يفشل النموذج في الازدحام، وهو أهم ما نحتاجه</td></tr>
<tr><td>نسيان تبديل الصيغة إلى YOLO</td>
    <td>ملفات XML لا تقرأها Ultralytics إطلاقاً</td></tr>
<tr><td>تغيير ترتيب الفئات بعد التوسيم</td>
    <td>كل ملفات التوسيم تصبح خاطئة صامتةً</td></tr>
</tbody>
</table>

<div class="note">
<p>الأخطاء الثلاثة الأخيرة في الجدول واجهناها فعلياً أثناء تجهيز قاعدة
بيانات هذا الدرس، وسنرى أثرها في القسم التالي.</p>
</div>

<h2>3.9 من أين نحصل على قواعد بيانات جاهزة؟</h2>

<p>توسيم قاعدة بيانات من الصفر عمل طويل. لحسن الحظ توجد مستودعات كبيرة
فيها قواعد بيانات موسومة جاهزة:</p>

<table>
<thead>
<tr><th>المصدر</th><th>ماذا يقدّم</th><th>كيف يعمل</th></tr>
</thead>
<tbody>
<tr>
  <td><a href="https://universe.roboflow.com/">Roboflow Universe</a></td>
  <td>عشرات الآلاف من قواعد بيانات الكشف الجاهزة بصيغة YOLO</td>
  <td>تختار مشروعاً، وتولّد منه نسخة (version) بالتعزيز الذي تريده، ثم
      تصدّرها بصيغة YOLOv8 عبر رابط مباشر أو حزمة <code>roboflow</code>
      مع مفتاح API مجاني</td>
</tr>
<tr>
  <td><a href="https://storage.googleapis.com/openimages/web/index.html">Open Images</a></td>
  <td>نحو 600 فئة بصناديق موسومة يدوياً على صور حقيقية، ومنها
      <code>Ambulance</code></td>
  <td>تنزّل ملفات التوسيم CSV وتصفّيها بالفئة، ثم تنزّل الصور المطلوبة
      فقط وتحوّلها إلى صيغة YOLO. بلا مفاتيح ولا تسجيل</td>
</tr>
<tr>
  <td><a href="https://www.kaggle.com/datasets">Kaggle Datasets</a></td>
  <td>مجموعات متنوعة جداً، لكن كثيراً منها للتصنيف لا للكشف</td>
  <td>تنزيل يدوي، أو عبر أداة <code>kaggle</code> بملف اعتماد
      <code>kaggle.json</code></td>
</tr>
<tr>
  <td><a href="https://huggingface.co/datasets">Hugging Face</a></td>
  <td>مرايا لمجموعات كثيرة ومجموعات مجتمعية</td>
  <td>عبر حزمة <code>huggingface_hub</code> أو تنزيل مباشر</td>
</tr>
<tr>
  <td><a href="https://cocodataset.org/">COCO</a></td>
  <td>الثمانون فئة العامة التي تدرّب عليها نموذجنا الجاهز</td>
  <td>مرجعي في درسنا: هو مصدر المشكلة لا الحل</td>
</tr>
</tbody>
</table>

<div class="note">
<p><strong>قبل أن تستخدم أي مجموعة، تحقق من أمرين:</strong></p>
<ol>
<li><strong>الرخصة.</strong> هل يُسمح بالاستخدام التعليمي؟ التجاري؟ هل
يجب نسب المصدر؟</li>
<li><strong>جودة التوسيم.</strong> افتح عشرين صورة وانظر إلى صناديقها
بعينك قبل أن تدرّب. عدد كبير من المجموعات المنشورة توسيمها رديء.</li>
</ol>
</div>

<h2>3.10 قاعدة بياناتنا: من أين جاءت وماذا فيها</h2>

<p>قاعدة بيانات هذا الدرس مستخرجة من <strong>Open Images V6/V7</strong>
من Google. اخترناها لأن:</p>

<ul>
<li>الفئة <code>Ambulance</code> موجودة فيها بصناديق موسومة يدوياً على
صور شوارع حقيقية.</li>
<li>الصور برخصة CC BY 2.0 والتوسيم برخصة CC BY 4.0، فيمكن إعادة نشرها
في مستودع تعليمي مفتوح.</li>
<li>التنزيل لا يحتاج حساباً ولا مفتاحاً.</li>
</ul>

<p>تفاصيل الرخصة وخطوات التحويل كاملة في <code>dataset/LICENSE.md</code>،
وسكربت التحويل في <code>tools/prepare_week7_dataset.py</code>.</p>

<p>لنفحصها برمجياً:</p>

</div>


In [ ]:
with open("dataset/data.yaml") as handle:
    data_config = yaml.safe_load(handle)

CLASS_NAMES = data_config["names"]
print("الفئات:", CLASS_NAMES)
print()

# نعدّ الصناديق لكل فئة في كل قسم
summary = {}

for split in ["train", "valid", "test"]:
    counts = Counter()
    for label_file in (Path("dataset") / split / "labels").glob("*.txt"):
        for line in label_file.read_text().split("\n"):
            if line.strip():
                counts[CLASS_NAMES[int(line.split()[0])]] += 1
    summary[split] = counts

distribution = pd.DataFrame(summary).fillna(0).astype(int)
distribution.loc["المجموع"] = distribution.sum()
distribution


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>لننظر أيضاً إلى أحجام الصناديق، فهي تخبرنا هل الأجسام كبيرة
وقريبة أم صغيرة وبعيدة، وهذا يؤثر على اختيار <code>imgsz</code> لاحقاً.</p>

</div>


In [ ]:
box_areas = []

for label_file in Path("dataset/train/labels").glob("*.txt"):
    for line in label_file.read_text().split("\n"):
        if line.strip():
            _, _, _, w, h = line.split()
            box_areas.append(float(w) * float(h))

box_areas = np.array(box_areas)

print(f"عدد الصناديق: {len(box_areas)}")
print(f"وسيط مساحة الصندوق: {np.median(box_areas) * 100:.1f}% من مساحة الصورة")
print(f"أصغر صندوق: {box_areas.min() * 100:.2f}%")
print(f"أكبر صندوق: {box_areas.max() * 100:.1f}%")

plt.figure(figsize=(9, 4))
plt.hist(np.sqrt(box_areas), bins=40, color="#1565C0")
plt.xlabel("Relative box size (sqrt of area fraction)")
plt.ylabel("Number of boxes")
plt.title("How big are the objects in our dataset?")
plt.show()


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>وأخيراً، أهم فحص على الإطلاق: أن ننظر إلى التوسيم بأعيننا. لا
تدرّب أبداً على قاعدة بيانات لم تر صناديقها.</p>

</div>


In [ ]:
BOX_COLORS = {
    "ambulance": (198, 40, 40),
    "car": (21, 101, 192),
    "truck": (46, 125, 50),
    "bus": (249, 168, 37),
}

sample_paths = sorted(Path("dataset/train/images").glob("*.jpg"))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, image_path in zip(axes.flat, sample_paths):
    image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]

    label_path = Path("dataset/train/labels") / (image_path.stem + ".txt")

    for line in label_path.read_text().split("\n"):
        if not line.strip():
            continue

        class_id, xc, yc, bw, bh = line.split()
        class_name = CLASS_NAMES[int(class_id)]
        xc, yc, bw, bh = float(xc), float(yc), float(bw), float(bh)

        p1 = (int((xc - bw / 2) * width), int((yc - bh / 2) * height))
        p2 = (int((xc + bw / 2) * width), int((yc + bh / 2) * height))

        cv2.rectangle(image, p1, p2, BOX_COLORS[class_name], 2)
        cv2.putText(
            image,
            class_name,
            (p1[0], max(p1[1] - 6, 12)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            BOX_COLORS[class_name],
            2,
            cv2.LINE_AA,
        )

    ax.imshow(image)
    ax.axis("off")

plt.suptitle("Ground truth annotations from our dataset", fontsize=16)
plt.tight_layout()
plt.show()


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>4. التعزيز Data Augmentation</h1>

<h2>4.1 لماذا نحتاجه؟</h2>

<p>قاعدة بياناتنا فيها 477 صورة تدريب فقط. لو عرضناها على النموذج كما هي
ستين مرة، فسيحفظها حفظاً بدل أن يتعلّم منها، وسيفشل على أول صورة جديدة.</p>

<p>التعزيز يحلّ هذا بأن يعرض النموذج نسخة معدَّلة قليلاً من الصورة في كل
مرة: مقلوبة، أو أفتح، أو مقصوصة، أو مصغَّرة. فيرى النموذج عملياً آلاف
الصور المختلفة من مئاتٍ قليلة، ويتعلّم أن سيارة الإسعاف تبقى سيارة إسعاف
مهما تغيّر لون الإضاءة أو زاوية اللقطة.</p>

<div class="note">
<p><strong>قاعدة أساسية:</strong> التعزيز يُطبَّق على الصورة
<strong>وعلى صناديقها معاً</strong>. لو قلبنا الصورة أفقياً ونسينا قلب
الصناديق، لأصبح كل توسيمنا خاطئاً. مكتبة Ultralytics تتولى هذا تلقائياً،
لكن إن كتبت تعزيزاً بنفسك فهذه مسؤوليتك.</p>
</div>

<h2>4.2 ما تفعله Ultralytics تلقائياً</h2>

<p>عند استدعاء <code>model.train()</code> تُطبَّق مجموعة تعزيزات افتراضية
دون أن تطلبها:</p>

<table>
<thead>
<tr><th>المعامل</th><th>القيمة الافتراضية</th><th>ماذا يفعل</th></tr>
</thead>
<tbody>
<tr><td><code>mosaic</code></td><td>1.0</td>
    <td>يدمج أربع صور في صورة واحدة، وهو الأقوى أثراً</td></tr>
<tr><td><code>fliplr</code></td><td>0.5</td>
    <td>يقلب الصورة أفقياً في نصف الحالات</td></tr>
<tr><td><code>scale</code></td><td>0.5</td>
    <td>يكبّر ويصغّر ليتعلّم النموذج أحجاماً مختلفة</td></tr>
<tr><td><code>translate</code></td><td>0.1</td>
    <td>يزيح الصورة قليلاً</td></tr>
<tr><td><code>hsv_h / hsv_s / hsv_v</code></td><td>0.015 / 0.7 / 0.4</td>
    <td>يغيّر التدرّج والتشبّع والإضاءة</td></tr>
<tr><td><code>erasing</code></td><td>0.4</td>
    <td>يمسح جزءاً عشوائياً ليتعلّم التعامل مع الحجب</td></tr>
</tbody>
</table>

<p>أقواها أثراً هو <strong>Mosaic</strong>. الشكل التالي يوضح كيف يعمل:</p>

</div>


<div style="text-align:center; margin:24px 0;">
<img
src="media/mosaic_augmentation.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week7/media/mosaic_augmentation.jpg';"
alt="Figure 7 - Mosaic augmentation"
width="900">
</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>4.3 لنرَ التعزيز على بياناتنا</h2>

<p>بدل أن نصدّق الكلام، لنطبّق بعض هذه التحويلات على صورة حقيقية من
قاعدة بياناتنا وننظر إلى النتيجة.</p>

</div>


In [ ]:
augmentation_source = sorted(Path("dataset/train/images").glob("*.jpg"))[3]
original = cv2.cvtColor(cv2.imread(str(augmentation_source)), cv2.COLOR_BGR2RGB)
height, width = original.shape[:2]

# قلب أفقي
flipped = original[:, ::-1]

# تغيير الإضاءة والتشبّع عبر فضاء HSV
hsv = cv2.cvtColor(original, cv2.COLOR_RGB2HSV).astype(np.int16)
hsv[..., 1] = np.clip(hsv[..., 1] * 1.6, 0, 255)   # تشبّع أعلى
hsv[..., 2] = np.clip(hsv[..., 2] * 0.55, 0, 255)  # إضاءة أخفض
darker = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

# تكبير مع قصّ من المركز
zoom = 1.4
zoomed = cv2.resize(original, None, fx=zoom, fy=zoom)
oy = (zoomed.shape[0] - height) // 2
ox = (zoomed.shape[1] - width) // 2
zoomed = zoomed[oy:oy + height, ox:ox + width]

# إزاحة
shift = np.float32([[1, 0, 0.12 * width], [0, 1, -0.08 * height]])
translated = cv2.warpAffine(original, shift, (width, height))

# مسح جزء عشوائي
erased = original.copy()
erased[int(0.35 * height):int(0.65 * height), int(0.30 * width):int(0.55 * width)] = 114

panels = [
    ("Original", original),
    ("Horizontal flip", flipped),
    ("HSV shift", darker),
    ("Scale + crop", zoomed),
    ("Translate", translated),
    ("Random erasing", erased),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, (title, image) in zip(axes.flat, panels):
    ax.imshow(image)
    ax.set_title(title, fontsize=13)
    ax.axis("off")

plt.suptitle("The same image, six ways the model will see it", fontsize=16)
plt.tight_layout()
plt.show()


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>4.4 التعزيز الذي يضرّ</h2>

<p>التعزيز ليس دائماً مفيداً. القاعدة: <strong>لا تولّد صوراً لا يمكن أن
تظهر في الواقع، ولا تغيّر ما يحمل المعنى.</strong></p>

<ul>
<li><strong>القلب الرأسي</strong> (<code>flipud</code>): سيارة مقلوبة رأساً
على عقب لا تحدث في الشارع. اتركه صفراً في مسائل المرور.</li>
<li><strong>القلب الأفقي مع النصوص</strong>: لو كنا نقرأ لوحات الأرقام أو
لافتات، فالقلب الأفقي يجعل الكتابة معكوسة ويعلّم النموذج شيئاً خاطئاً.
في حالتنا نحن نكشف المركبة كاملة لا نقرأ نصاً، فالقلب مفيد.</li>
<li><strong>تغيير الألوان بإفراط</strong>: سيارة الإسعاف تُعرف جزئياً
بلونها الأبيض وعلاماتها الحمراء. لو غيّرنا التدرّج
(<code>hsv_h</code>) كثيراً لمحونا إشارة مهمة يعتمد عليها النموذج.</li>
<li><strong>التدوير الكبير</strong>: مفيد في صور الأقمار الصناعية، ضار في
كاميرا مثبّتة على عمود.</li>
</ul>

<blockquote>
<p>اسأل نفسك دائماً: هل يمكن أن تصل هذه الصورة من الكاميرا الحقيقية؟ إن
كان الجواب لا، فالتعزيز يضرّ لا ينفع.</p>
</blockquote>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>5. تشريح عملية التدريب</h1>

<p>قبل أن نضغط زر التدريب، لنفهم ماذا يحدث بالضبط ومعنى كل رقم سنراه.</p>

<h2>5.1 Epoch و Batch و Iteration</h2>

<p>ثلاثة مصطلحات يخلط بينها كثيرون:</p>

<ul>
<li><strong>Batch</strong>: عدد الصور التي يعالجها النموذج دفعة واحدة قبل
أن يعدّل أوزانه مرة. مقيَّد بحجم الذاكرة.</li>
<li><strong>Iteration</strong>: تعديل واحد للأوزان، أي معالجة batch واحد.</li>
<li><strong>Epoch</strong>: مرور كامل على كل صور التدريب مرة واحدة.</li>
</ul>

<p>بالأرقام في تدريبنا المصغّر:</p>

</div>


In [ ]:
MINI_TRAIN_IMAGES = 100
MINI_BATCH = 8
MINI_EPOCHS = 5

iterations_per_epoch = -(-MINI_TRAIN_IMAGES // MINI_BATCH)   # قسمة لأعلى

print(f"صور التدريب      : {MINI_TRAIN_IMAGES}")
print(f"حجم الـ batch     : {MINI_BATCH}")
print(f"iterations في كل epoch : {iterations_per_epoch}")
print(f"عدد الـ epochs    : {MINI_EPOCHS}")
print(f"إجمالي تعديلات الأوزان : {iterations_per_epoch * MINI_EPOCHS}")


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>5.2 خسائر YOLO الثلاث</h2>

<p>أثناء التدريب ستظهر ثلاثة أرقام للخسارة، لأن الكشف ثلاث مسائل في آن
واحد:</p>

<table>
<thead>
<tr><th>الخسارة</th><th>تقيس</th><th>حين ترتفع فهذا يعني</th></tr>
</thead>
<tbody>
<tr><td><code>box_loss</code></td>
    <td>دقة موضع الصندوق وحجمه</td>
    <td>النموذج يجد الأجسام لكن صناديقه غير دقيقة</td></tr>
<tr><td><code>cls_loss</code></td>
    <td>صحة الفئة المختارة</td>
    <td>النموذج يجد الأجسام لكنه يخطئ في تسميتها</td></tr>
<tr><td><code>dfl_loss</code></td>
    <td>دقة توزيع حدود الصندوق (Distribution Focal Loss)</td>
    <td>حواف الصناديق غير محسومة</td></tr>
</tbody>
</table>

<p>الفصل بينها مفيد عملياً: لو كانت <code>cls_loss</code> عالية و
<code>box_loss</code> منخفضة، فمشكلتك في التمييز بين الفئات لا في
تحديد المواقع، والحل يكون بمزيد من الأمثلة المتنوعة لكل فئة لا بتحسين
التوسيم.</p>

<h2>5.3 معدّل التعلّم Learning Rate والإحماء Warmup</h2>

<p>معدّل التعلّم يحدّد حجم الخطوة عند تعديل الأوزان:</p>

<ul>
<li><strong>كبير جداً</strong>: يقفز فوق الحل الأمثل، والخسارة تتذبذب أو
تنفجر.</li>
<li><strong>صغير جداً</strong>: يتعلّم ببطء شديد، وقد يعلق في حل رديء.</li>
</ul>

<p>في Fine-Tuning نستخدم معدّلاً <strong>أصغر</strong> من التدريب من
الصفر، لأننا نبدأ من أوزان جيدة ولا نريد تخريبها. القيمة الافتراضية في
Ultralytics <code>lr0=0.01</code> وهي مناسبة عادة.</p>

<p>و<strong>الإحماء</strong> (<code>warmup_epochs=3</code>) يبدأ بمعدّل
صغير جداً ويرفعه تدريجياً في أول عدة epochs. السبب أن الرأس Head مهيّأ
عشوائياً في البداية، فلو ضربناه بخطوة كبيرة فوراً لأفسد الأوزان الجيدة
القادمة من الـ Backbone.</p>

<h2>5.4 imgsz و batch وحدود الذاكرة</h2>

<p><code>imgsz</code> هو الحجم الذي تُصغَّر إليه كل صورة قبل دخولها
النموذج. أثره كبير على السرعة وعلى كشف الأجسام الصغيرة:</p>

<table>
<thead>
<tr><th>الوضع</th><th>imgsz</th><th>batch</th><th>ملاحظة</th></tr>
</thead>
<tbody>
<tr><td>معالج فقط، داخل الحصة</td><td>320</td><td>8</td>
    <td>ما سنستخدمه الآن. سريع، ودقته متواضعة</td></tr>
<tr><td>كرت شاشة 6 GB</td><td>640</td><td>16</td>
    <td>الوضع المتوازن المعتاد</td></tr>
<tr><td>كرت شاشة 12 GB أو أكثر</td><td>640</td><td>32</td>
    <td>أسرع، ونتائج أثبت</td></tr>
<tr><td>أجسام صغيرة وبعيدة</td><td>960</td><td>8</td>
    <td>ذاكرة أكبر بكثير وزمن أطول</td></tr>
</tbody>
</table>

<div class="note">
<p>إن ظهرت رسالة <code>CUDA out of memory</code> فالحل هو تصغير
<code>batch</code> أولاً، ثم تصغير <code>imgsz</code>. لاحظ أن
<code>imgsz</code> يجب أن يكون من مضاعفات 32.</p>
</div>

<p>وسيط بياناتنا يقول إن الأجسام تشغل نحو 12% من مساحة الصورة، وهي أجسام
كبيرة نسبياً، لذلك <code>imgsz=320</code> يكفي في الحصة ولن نخسر كثيراً.</p>

<h2>5.5 تجميد الطبقات Freezing</h2>

<p>يمكننا أن نمنع الطبقات الأولى من التعديل ونكتفي بتدريب الرأس:</p>

<pre><code>model.train(data=..., freeze=10)   # تجميد أول عشر طبقات</code></pre>

<ul>
<li><strong>متى يفيد؟</strong> حين تكون البيانات قليلة جداً (أقل من مئة
صورة)، أو حين نريد تدريباً أسرع بكثير على معالج ضعيف.</li>
<li><strong>متى يضرّ؟</strong> حين تختلف صورنا كثيراً عن ImageNet و COCO،
لأن الـ Backbone عندها يحتاج فعلاً أن يتكيّف.</li>
</ul>

<p>في حالتنا صور شوارع عادية، والـ Backbone مناسب أصلاً، فالتجميد خيار
معقول لتوفير الوقت. سنتركه مفتوحاً في التدريب الكامل، وستجرّبه أنت في
التمارين.</p>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>6. لنُدرّب</h1>

<p>سنقوم بتدريبين مختلفين تماماً في الهدف:</p>

<ol>
<li><strong>تدريب مصغّر حي</strong> تشغّله الآن على معالج جهازك خلال بضع
دقائق. هدفه أن ترى الخسارة تنخفض بعينك وتفهم مجرى العملية.
<strong>نتيجته ستكون ضعيفة، وهذا مقصود.</strong></li>
<li><strong>تدريب كامل</strong> أُجري مسبقاً على كرت شاشة، ونتائجه مرفقة
في مجلد <code>runs_reference/</code>. عليه سنبني كل التحليل في القسم
السابع.</li>
</ol>

<h2>6.1 نبني قاعدة بيانات مصغّرة للحصة</h2>

<p>477 صورة على معالج ستأخذ وقتاً طويلاً. سنأخذ عيّنة صغيرة منها فقط.</p>

</div>


In [ ]:
import random
import shutil

random.seed(7)

MINI_ROOT = Path("dataset_mini")
if MINI_ROOT.exists():
    shutil.rmtree(MINI_ROOT)

for split, n_images in [("train", MINI_TRAIN_IMAGES), ("valid", 30)]:
    source_images = sorted((Path("dataset") / split / "images").glob("*.jpg"))
    chosen = random.sample(source_images, min(n_images, len(source_images)))

    (MINI_ROOT / split / "images").mkdir(parents=True, exist_ok=True)
    (MINI_ROOT / split / "labels").mkdir(parents=True, exist_ok=True)

    for image_path in chosen:
        label_path = Path("dataset") / split / "labels" / (image_path.stem + ".txt")
        shutil.copy2(image_path, MINI_ROOT / split / "images" / image_path.name)
        shutil.copy2(label_path, MINI_ROOT / split / "labels" / label_path.name)

    print(f"{split}: نسخنا {len(chosen)} صورة")

# ملف إعداد خاص بالنسخة المصغّرة، بالقاعدة نفسها: بلا مفتاح path، فيصير
# مجلد هذا الملف هو الجذر.
(MINI_ROOT / "data.yaml").write_text(
    "train: train/images\n"
    "val: valid/images\n\n"
    f"nc: {len(CLASS_NAMES)}\n"
    f"names: {CLASS_NAMES}\n"
)

print("\nجاهز:", MINI_ROOT / "data.yaml")


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>6.2 التدريب المصغّر</h2>

<p>الخلية التالية تدرّب فعلياً. على معالج بأربعة أنوية تستغرق نحو خمس
دقائق. راقب عمود <code>cls_loss</code>: هو الأوضح انخفاضاً، لأن مسألتنا
الأساسية هي تعليم النموذج <strong>اسماً جديداً</strong> لا تحسين دقة
صناديقه.</p>

<div class="note">
<p><code>amp=False</code> ضرورية هنا لسببين: الحساب نصف الدقة لا يعمل على
المعالج أصلاً، والأهم أن فحص AMP في Ultralytics يحاول تحميل نموذج من
الإنترنت، ونحن نعمل بلا اتصال.</p>
</div>

</div>


In [ ]:
mini_model = YOLO(PRETRAINED_PATH)

mini_results = mini_model.train(
    data=str(MINI_ROOT / "data.yaml"),
    epochs=MINI_EPOCHS,
    imgsz=320,
    batch=MINI_BATCH,
    device=DEVICE,
    workers=0,
    amp=False,
    cache=False,
    plots=True,
    name="mini",
    exist_ok=True,
    seed=7,
)

# لا نخمّن مسار المخرجات، بل نسأل المدرِّب عنه. المسار يختلف بين إصدارات
# Ultralytics وبحسب إعداداتها المحلية.
MINI_RUN_DIR = Path(mini_model.trainer.save_dir)

print("\nانتهى التدريب المصغّر.")
print("مخرجاته في:", MINI_RUN_DIR)


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>6.3 ماذا تعني أرقام شريط التقدم؟</h2>

<p>أثناء التدريب رأيت سطراً يشبه هذا:</p>

<pre><code>  Epoch  GPU_mem  box_loss  cls_loss  dfl_loss  Instances  Size
    3/5       0G     1.842     2.104     1.203         21   320</code></pre>

<ul>
<li><code>3/5</code>: نحن في الدورة الثالثة من خمس.</li>
<li><code>box_loss</code> و <code>cls_loss</code> و <code>dfl_loss</code>:
الخسائر الثلاث التي شرحناها. <strong>يجب أن تنخفض مع الوقت.</strong></li>
<li><code>Instances</code>: عدد الأجسام في الـ batch الحالي، لا عدد
الصور.</li>
<li><code>Size</code>: حجم الصورة الداخل إلى النموذج، أي
<code>imgsz</code>.</li>
</ul>

<p>وبعد كل epoch يظهر سطر تقييم على مجموعة <code>valid</code> فيه
<code>Box(P R mAP50 mAP50-95)</code>، وهي المقاييس التي سنشرحها في القسم
التالي.</p>

<h2>6.4 نتيجة تدريبنا المصغّر</h2>

</div>


In [ ]:
mini_history = pd.read_csv(MINI_RUN_DIR / "results.csv")
mini_history.columns = [c.strip() for c in mini_history.columns]

loss_columns = [c for c in mini_history.columns if "train/" in c and "loss" in c]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for column in loss_columns:
    axes[0].plot(mini_history["epoch"], mini_history[column],
                 marker="o", label=column.replace("train/", ""))
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].set_title("Training losses (mini run)")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(mini_history["epoch"], mini_history["metrics/mAP50(B)"],
             marker="o", color="#2E7D32")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("mAP@50")
axes[1].set_title("Validation mAP@50 (mini run)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("mAP@50 بعد التدريب المصغّر:",
      round(float(mini_history["metrics/mAP50(B)"].iloc[-1]), 3))


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<div class="note">
<p><strong>هل انخفضت كل الخسائر؟ غالباً لا، وهذا طبيعي.</strong> سترى
<code>cls_loss</code> ينخفض بوضوح، بينما قد يرتفع <code>box_loss</code>
قليلاً في الدورات الأولى. السبب أن تعزيز <strong>Mosaic</strong> الذي
شرحناه في القسم الرابع يبني صوراً مركّبة أصعب بكثير من الصور الأصلية،
فيصبح ضبط الصناديق أصعب مؤقتاً قبل أن يتحسّن. في تدريب طويل يستقر
الاثنان معاً وينخفضان.</p>
<p><strong>المؤشر الأهم في تدريب قصير كهذا هو
<code>mAP@50</code></strong>: انظر هل ارتفع من الصفر. هذا وحده يكفي
دليلاً على أن النموذج بدأ يتعلّم الفئة الجديدة.</p>
</div>

<div class="note">
<p><strong>ولماذا النتيجة ضعيفة عموماً؟</strong> لأننا استخدمنا مئة صورة فقط، وخمس
دورات فقط، وحجم صورة 320 بدل 640. هذا ليس فشلاً بل هو حدود ما يمكن فعله
في دقائق على معالج. الغرض كان أن ترى الآلية تعمل: الخسارة تنخفض، والـ
mAP يرتفع من الصفر.</p>
<p>لتحصل على نموذج مفيد فعلاً نحتاج بيانات أكثر ودورات أكثر وحجم صورة
أكبر، وهذا ما فعلناه في التدريب المرجعي.</p>
</div>

<h2>6.5 التدريب الكامل</h2>

<p>الخلية التالية معطّلة بثابت. شغّلها في البيت إن كان لديك كرت شاشة
NVIDIA. تستغرق نحو نصف ساعة.</p>

</div>


In [ ]:
RUN_FULL_TRAINING = False   # اجعلها True إن كان لديك كرت شاشة NVIDIA

if RUN_FULL_TRAINING:
    full_model = YOLO(PRETRAINED_PATH)

    full_model.train(
        data=DATA_YAML,
        epochs=60,
        imgsz=640,
        batch=16,
        device=DEVICE,
        workers=4,
        amp=True,
        patience=15,
        plots=True,
        name="emergency_full",
        exist_ok=True,
        seed=7,
    )

    print("مخرجات التدريب الكامل في:", full_model.trainer.save_dir)
else:
    print("التدريب الكامل معطّل.")
    print("سنستخدم النتائج المرفقة في", REFERENCE_RUN)


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>6.6 النتائج المرجعية المرفقة</h2>

<p>مجلد <code>runs_reference/emergency_full/</code> يحوي مخرجات تدريب كامل
أجريناه مسبقاً على كامل قاعدة البيانات. سنعتمد عليه في كل التحليل التالي.</p>

</div>


In [ ]:
for item in sorted(REFERENCE_RUN.rglob("*")):
    if item.is_file():
        size_kb = item.stat().st_size / 1024
        print(f"{str(item.relative_to(REFERENCE_RUN)):38s} {size_kb:8.1f} KB")


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>7. قراءة النتائج والتقييم</h1>

<div class="note">
<p><strong>تنبيه مهم:</strong> كل الأرقام في هذا القسم تأتي من
<strong>التدريب الكامل المرجعي</strong> الذي أُجري مسبقاً على كرت شاشة،
لا من تدريبك المصغّر قبل قليل. تدريبك المصغّر سيعطي أرقاماً أقل بكثير،
وهذا متوقع.</p>
</div>

<h2>7.1 مراجعة سريعة: IoU</h2>

<p>قبل أي مقياس، نحتاج أن نقرّر متى نعتبر صندوقاً متوقَّعاً
<strong>صحيحاً</strong>. المعيار هو <strong>IoU</strong>، أي نسبة التقاطع
إلى الاتحاد بين الصندوق المتوقَّع والصندوق الحقيقي.</p>

<p>القيمة تتراوح بين صفر وواحد، والعتبة الشائعة <code>0.5</code>: إن كان
التقاطع نصف الاتحاد أو أكثر فالكشف صحيح.</p>

</div>


In [ ]:
def intersection_over_union(box_a, box_b):
    """كل صندوق بصيغة (x1, y1, x2, y2)."""
    inter_x1 = max(box_a[0], box_b[0])
    inter_y1 = max(box_a[1], box_b[1])
    inter_x2 = min(box_a[2], box_b[2])
    inter_y2 = min(box_a[3], box_b[3])

    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union_area = area_a + area_b - inter_area

    return inter_area / union_area if union_area > 0 else 0.0


ground_truth = (100, 100, 300, 250)

for name, prediction in [
    ("مطابق تماماً", (100, 100, 300, 250)),
    ("إزاحة بسيطة", (110, 108, 305, 258)),
    ("إزاحة كبيرة", (180, 150, 380, 300)),
    ("بعيد تماماً", (320, 260, 480, 380)),
]:
    score = intersection_over_union(ground_truth, prediction)
    verdict = "مقبول" if score >= 0.5 else "مرفوض"
    print(f"{name:14s} IoU = {score:.2f}  ->  {verdict}")


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>7.2 TP و FP و FN ثم Precision و Recall</h2>

<p>بعد تطبيق عتبة IoU يصبح كل كشف واحداً من ثلاثة:</p>

<ul>
<li><strong>TP</strong> (True Positive): كشف صحيح طابق جسماً حقيقياً.</li>
<li><strong>FP</strong> (False Positive): كشف على لا شيء، أو بفئة خاطئة.
إنذار كاذب.</li>
<li><strong>FN</strong> (False Negative): جسم حقيقي لم يكشفه النموذج.
فوّتناه.</li>
</ul>

<p>ومنها مقياسان:</p>

<ul>
<li><strong>Precision</strong> = TP / (TP + FP)
  — «مما أعلنتُ عنه، كم كان صحيحاً؟»</li>
<li><strong>Recall</strong> = TP / (TP + FN)
  — «مما كان موجوداً فعلاً، كم وجدتُ؟»</li>
</ul>

<p>بينهما مقايضة دائمة: خفض <code>conf</code> يرفع الـ Recall ويخفض الـ
Precision، ورفعه يفعل العكس. وهذا بالضبط ما جرّبناه في الأسبوع الماضي.</p>

<div class="note">
<p><strong>في نظامنا أيهما أهم؟</strong> إن فاتنا إسعاف
(<strong>FN</strong>) فلن تُفتح له الإشارة وقد يتأخر عن حالة طارئة. وإن
أطلقنا إنذاراً كاذباً (<strong>FP</strong>) فسنعطّل حركة المرور بلا سبب.
الخطأ الأول أخطر، لذلك سنميل إلى <strong>Recall</strong> أعلى ونتحمّل
بعض الإنذارات الكاذبة.</p>
</div>

</div>


<div style="text-align:center; margin:24px 0;">
<img
src="media/tp_fp_fn.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week7/media/tp_fp_fn.jpg';"
alt="Figure 8 - True Positive / False Positive / False Negative"
width="900">
</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>7.3 منحنى PR و mAP</h2>

<p>مشكلة Precision و Recall أن كلاً منهما يعتمد على عتبة
<code>conf</code> التي اخترناها. فأي قيمة نُبلغ عنها؟</p>

<p>الحل أن نجرّب <strong>كل</strong> العتبات ونرسم Precision مقابل Recall،
فنحصل على <strong>منحنى PR</strong>. والمساحة تحت هذا المنحنى هي
<strong>Average Precision (AP)</strong> لتلك الفئة. ومتوسط AP على كل
الفئات هو <strong>mAP</strong>.</p>

<ul>
<li><strong>mAP@50</strong>: بعتبة IoU واحدة هي 0.5. متساهل نسبياً، يقيس
أساساً «هل وجدت الجسم وسمّيته صحيحاً؟»</li>
<li><strong>mAP@50-95</strong>: متوسط عشر عتبات من 0.50 إلى 0.95. أقسى
بكثير، ويكافئ دقة الصندوق لا مجرد إيجاد الجسم. هذا هو الرقم الذي يُنشر
في الأبحاث.</li>
</ul>

<p>ستكون <code>mAP@50-95</code> دائماً أقل من <code>mAP@50</code>. لا
تقارن رقماً من نوع برقم من نوع آخر.</p>

</div>


<div style="text-align:center; margin:24px 0;">
<img
src="media/pr_curve_map.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week7/media/pr_curve_map.jpg';"
alt="Figure 9 - PR curve and mAP"
width="900">
</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>7.4 كيف نقرأ results.png</h2>

<p>تنتج Ultralytics بعد كل تدريب صورة تلخّص المسيرة كاملة. لنعرضها من
التدريب المرجعي:</p>

</div>


In [ ]:
from IPython.display import Image as IPImage

IPImage(str(REFERENCE_RUN / "results.png"), width=1100)


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>الصورة عشر لوحات. اقرأها هكذا:</p>

<ul>
<li><strong>الصف العلوي</strong>: خسائر التدريب الثلاث
(<code>train/box_loss</code>, <code>cls_loss</code>, <code>dfl_loss</code>)
ثم مقياسا <code>precision</code> و <code>recall</code>.</li>
<li><strong>الصف السفلي</strong>: الخسائر الثلاث نفسها لكن على مجموعة
<code>valid</code>، ثم <code>mAP50</code> و <code>mAP50-95</code>.</li>
</ul>

<p>ما تبحث عنه:</p>

<ol>
<li>خسائر التدريب تنخفض بسلاسة ← معدّل التعلّم مناسب.</li>
<li>خسائر <code>valid</code> تنخفض معها ← النموذج يعمّم لا يحفظ.</li>
<li>منحنيات mAP ترتفع ثم تستوي ← اقتربنا من حدود البيانات الحالية.</li>
<li>لو ارتفعت خسارة <code>valid</code> بينما تواصل خسارة التدريب الانخفاض
← هذا هو <strong>Overfitting</strong> بعينه.</li>
</ol>

<p>ولنقرأ الأرقام النهائية مباشرة من الملف:</p>

</div>


In [ ]:
reference_history = pd.read_csv(REFERENCE_RUN / "results.csv")
reference_history.columns = [c.strip() for c in reference_history.columns]

final = reference_history.iloc[-1]

print(f"عدد الدورات المنفَّذة : {int(final['epoch'])}")
print(f"Precision          : {final['metrics/precision(B)']:.3f}")
print(f"Recall             : {final['metrics/recall(B)']:.3f}")
print(f"mAP@50             : {final['metrics/mAP50(B)']:.3f}")
print(f"mAP@50-95          : {final['metrics/mAP50-95(B)']:.3f}")

best_epoch = int(reference_history["metrics/mAP50-95(B)"].idxmax()) + 1
print(f"\nأفضل دورة حسب mAP@50-95 : {best_epoch}")
print("وهي الدورة المحفوظة في best.pt")


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>7.5 مصفوفة الالتباس Confusion Matrix</h2>

<p>الـ mAP رقم واحد يلخّص كل شيء، وهذا عيبه: لا يخبرنا <strong>أين</strong>
يخطئ النموذج. مصفوفة الالتباس تجيب عن ذلك.</p>

</div>


In [ ]:
IPImage(str(REFERENCE_RUN / "confusion_matrix_normalized.png"), width=850)


<div style="text-align:center; margin:24px 0;">
<img
src="media/confusion_matrix_guide.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week7/media/confusion_matrix_guide.jpg';"
alt="Figure 10 - Confusion matrix guide"
width="900">
</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>الأسئلة التي تجيب عنها المصفوفة:</p>

<ul>
<li><strong>القطر</strong>: نسبة الإصابة لكل فئة. كلما اقترب من 1 كان
أفضل.</li>
<li><strong>خلية خارج القطر</strong>: التبس على النموذج بين فئتين. راقب
خانة <code>ambulance</code> مقابل <code>truck</code> بالذات، فهي الخطأ
الذي بدأنا منه الدرس.</li>
<li><strong>عمود background</strong>: أجسام حقيقية لم يكشفها النموذج
إطلاقاً، أي FN.</li>
<li><strong>صف background</strong>: صناديق رسمها النموذج على خلفية فارغة،
أي FP.</li>
</ul>

<h2>7.6 هل عندنا Overfitting؟</h2>

<p><strong>Overfitting</strong> أن يحفظ النموذج صور التدريب بدل أن يتعلّم
منها قاعدة عامة. علاماته:</p>

<ul>
<li>خسارة التدريب تواصل الانخفاض، وخسارة <code>valid</code> تتوقف ثم
ترتفع.</li>
<li>فجوة كبيرة ومتّسعة بين أداء <code>train</code> و <code>valid</code>.</li>
<li>أداء ممتاز على بياناتك وفشل ذريع على أي صورة جديدة.</li>
</ul>

<p>وعلاجه بالترتيب العملي:</p>

<ol>
<li><strong>بيانات أكثر وأكثر تنوّعاً</strong> — الحل الأقوى دائماً.</li>
<li><strong>تعزيز أقوى</strong> — أرخص وأسرع من جمع بيانات.</li>
<li><strong>إيقاف مبكر</strong> (<code>patience</code>) — Ultralytics
تفعله تلقائياً وتحفظ أفضل نسخة في <code>best.pt</code>.</li>
<li><strong>تجميد طبقات</strong> (<code>freeze</code>) — يقلّل عدد
الأوزان القابلة للتعديل.</li>
<li><strong>نموذج أصغر</strong> — <code>yolo11n</code> بدل
<code>yolo11s</code>.</li>
</ol>

</div>


<div style="text-align:center; margin:24px 0;">
<img
src="media/overfitting_curves.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week7/media/overfitting_curves.jpg';"
alt="Figure 11 - Overfitting curves"
width="900">
</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>ولنفحص منحنياتنا نحن: هل الفجوة بين التدريب والتحقق تتّسع؟</p>

</div>


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(reference_history["epoch"], reference_history["train/box_loss"],
        label="train/box_loss", color="#1565C0", linewidth=2)
ax.plot(reference_history["epoch"], reference_history["val/box_loss"],
        label="val/box_loss", color="#F9A825", linewidth=2)

ax.set_xlabel("epoch")
ax.set_ylabel("box loss")
ax.set_title("Train vs validation loss - are they drifting apart?")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

gap_start = float(reference_history["val/box_loss"].iloc[0]
                  - reference_history["train/box_loss"].iloc[0])
gap_end = float(reference_history["val/box_loss"].iloc[-1]
                - reference_history["train/box_loss"].iloc[-1])

print(f"الفجوة في البداية : {gap_start:.3f}")
print(f"الفجوة في النهاية : {gap_end:.3f}")
print("الفجوة تتّسع، وهي علامة تجهيز زائد."
      if gap_end > gap_start else "الفجوة مستقرة، والوضع سليم.")


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>8. اللحظة الحاسمة: قبل وبعد</h1>

<p>نعود الآن إلى صور القسم الأول نفسها، تلك التي سمّى فيها النموذج الجاهز
سيارة الإسعاف <code>truck</code> بثقة تجاوزت 0.90. لنشغّل عليها النموذجين
جنباً إلى جنب.</p>

</div>


In [ ]:
finetuned_model = YOLO("models/emergency_best.pt")

n_failures = len(failures_df)
fig, axes = plt.subplots(2, n_failures, figsize=(4.2 * n_failures, 9))

for column, (_, item) in enumerate(failures_df.iterrows()):
    image_path = f"failures/{item['file']}"

    before = pretrained_model.predict(
        source=image_path, conf=0.35, device=DEVICE, verbose=False
    )[0]
    after = finetuned_model.predict(
        source=image_path, conf=0.35, device=DEVICE, verbose=False
    )[0]

    axes[0, column].imshow(cv2.cvtColor(before.plot(), cv2.COLOR_BGR2RGB))
    axes[0, column].axis("off")

    axes[1, column].imshow(cv2.cvtColor(after.plot(), cv2.COLOR_BGR2RGB))
    axes[1, column].axis("off")

axes[0, 0].set_ylabel("BEFORE")
axes[1, 0].set_ylabel("AFTER")

fig.text(0.5, 0.97, "Before: pretrained yolo11n (COCO, 80 classes)",
         ha="center", fontsize=15)
fig.text(0.5, 0.49, "After: fine-tuned on our 4 classes",
         ha="center", fontsize=15)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>والآن الأرقام. نقيّم النموذج المدرَّب على مجموعة
<code>test</code> التي لم يرَها إطلاقاً أثناء التدريب:</p>

</div>


In [ ]:
metrics = finetuned_model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    device=DEVICE,
    verbose=False,
)

per_class = []
for index, class_id in enumerate(metrics.ap_class_index):
    per_class.append({
        "الفئة": CLASS_NAMES[class_id],
        "Precision": round(float(metrics.box.p[index]), 3),
        "Recall": round(float(metrics.box.r[index]), 3),
        "mAP@50": round(float(metrics.box.ap50[index]), 3),
        "mAP@50-95": round(float(metrics.box.ap[index]), 3),
    })

results_table = pd.DataFrame(per_class)
print(results_table.to_string(index=False))
print()
print(f"المتوسط العام mAP@50    : {metrics.box.map50:.3f}")
print(f"المتوسط العام mAP@50-95 : {metrics.box.map:.3f}")


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<div class="note">
<p><strong>لماذا لا نضع جدولاً مقابلاً للنموذج الجاهز؟</strong> لأن
المقارنة العددية المباشرة مستحيلة أصلاً، وهذا هو بيت القصيد: النموذج
الجاهز لا يملك فئة <code>ambulance</code>، فقيمة
<code>mAP</code> له على هذه الفئة تساوي <strong>صفراً بالتعريف</strong>،
لا لأنه سيّئ بل لأن السؤال خارج قاموسه تماماً.</p>
<p>هذا هو الفرق بين «نموذج ضعيف» و«نموذج لا يستطيع التعبير عن المسألة».
والتدريب هو الطريق الوحيد من الثاني إلى الأول.</p>
</div>

<p>لاحظ أيضاً أن النموذج ما زال يكشف <code>car</code> و <code>truck</code>
و <code>bus</code>. لم نخسر شيئاً، بل أضفنا فئة رابعة يحتاجها نظامنا.</p>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>9. دليل التحسين حين تكون النتائج ضعيفة</h1>

<p>لن يكون تدريبك الأول جيداً غالباً. هذا الجدول يربط العَرَض بالسبب
المحتمل بالإجراء:</p>

<table>
<thead>
<tr><th>العَرَض</th><th>السبب المحتمل</th><th>الإجراء</th></tr>
</thead>
<tbody>
<tr><td>الخسارة لا تنخفض إطلاقاً</td>
    <td>خطأ في مسارات البيانات أو التوسيم</td>
    <td>افحص <code>labels.jpg</code>: هل الصناديق في مواضعها؟</td></tr>
<tr><td>الخسارة تنفجر أو تصبح NaN</td>
    <td>معدّل التعلّم كبير</td>
    <td>خفّض <code>lr0</code> إلى 0.001</td></tr>
<tr><td>mAP مرتفع للتدريب ومنخفض للتحقق</td>
    <td>تجهيز زائد Overfitting</td>
    <td>بيانات أكثر، تعزيز أقوى، <code>freeze</code></td></tr>
<tr><td>فئة واحدة أداؤها سيّئ وحدها</td>
    <td>أمثلتها قليلة أو توسيمها غير متسق</td>
    <td>أضف صوراً لها، وراجع توسيمها</td></tr>
<tr><td>Recall منخفض والـ Precision مرتفع</td>
    <td>النموذج متحفّظ جداً</td>
    <td>خفّض <code>conf</code>، وأضف أمثلة صعبة</td></tr>
<tr><td>Precision منخفض والـ Recall مرتفع</td>
    <td>إنذارات كاذبة كثيرة</td>
    <td>ارفع <code>conf</code>، وأضف صوراً سلبية</td></tr>
<tr><td>يفشل على الأجسام الصغيرة فقط</td>
    <td><code>imgsz</code> صغير</td>
    <td>ارفعه إلى 960 مع تصغير <code>batch</code></td></tr>
<tr><td>ممتاز على بياناتك وسيّئ في الواقع</td>
    <td>اختلاف المجال Domain Shift</td>
    <td>اجمع صوراً من الكاميرا الحقيقية نفسها</td></tr>
<tr><td>الفئتان تلتبسان دائماً</td>
    <td>الفئتان متشابهتان بصرياً فعلاً</td>
    <td>أعد التفكير: هل يجب فصلهما أصلاً؟</td></tr>
<tr><td>النتائج تتغيّر كل مرة</td>
    <td>لم تثبّت البذرة العشوائية</td>
    <td>مرّر <code>seed</code> ثابتاً</td></tr>
</tbody>
</table>

<hr />
<h1>10. حفظ النموذج واستخدامه لاحقاً</h1>

<p>ينتج التدريب ملفين في <code>runs/&lt;name&gt;/weights/</code>:</p>

<ul>
<li><code>best.pt</code>: أفضل نسخة حسب أداء <code>valid</code>.
<strong>هذا ما تستخدمه دائماً.</strong></li>
<li><code>last.pt</code>: آخر دورة. يفيد فقط لمتابعة تدريب انقطع.</li>
</ul>

</div>


In [ ]:
saved_model_path = Path("models/emergency_best.pt")

print("حجم ملف النموذج:", round(saved_model_path.stat().st_size / 1e6, 1), "MB")
print()

# الملف يحمل معه أسماء الفئات، فلا تحتاج data.yaml عند الاستخدام
loaded = YOLO(str(saved_model_path))
print("الفئات المحفوظة داخل النموذج:", loaded.names)


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>لتشغيله في مشروع آخر لا تحتاج إلا هذا الملف:</p>

<pre><code>from ultralytics import YOLO

model = YOLO("emergency_best.pt")
results = model.predict("street.jpg", conf=0.35)</code></pre>

<p>ولنشره على أجهزة صغيرة أو خوادم، يمكن تصديره إلى صيغ أخرى مثل
<strong>ONNX</strong> عبر <code>model.export(format="onnx")</code>، وهي
صيغة تعمل بلا PyTorch وأسرع على المعالج. لن نحتاجها في هذا المسار، لكن
اعرف أنها موجودة.</p>

<div class="note">
<p>احفظ <code>emergency_best.pt</code> في مكان آمن. سنستخدمه في الأسبوع
القادم لتتبّع سيارات الإسعاف عبر إطارات الفيديو.</p>
</div>

<hr />
<h1>11. حدود ما فعلناه</h1>

<p>من الأمانة العلمية أن نعرف حدود نموذجنا قبل أن نثق به:</p>

<ul>
<li><strong>قاعدة البيانات صغيرة.</strong> 682 صورة فقط، منها نحو 400
فيها إسعاف. النماذج الإنتاجية تُدرَّب على عشرات الآلاف.</li>
<li><strong>تحيّز جغرافي.</strong> معظم الصور من أوروبا وأمريكا الشمالية.
سيارات الإسعاف عندنا قد تختلف شكلاً ولوناً وعلامات، وأداء النموذج عليها
سيكون أضعف مما تُظهره أرقامنا.</li>
<li><strong>اختلاف المجال.</strong> صورنا لقطات فوتوغرافية من مستوى
الأرض، بينما كاميرا المرور مثبّتة على عمود عالٍ وبزاوية مختلفة وجودة
أقل.</li>
<li><strong>ظروف ناقصة.</strong> الصور نهارية في معظمها. الليل والمطر
والضباب ممثَّلة تمثيلاً ضعيفاً.</li>
<li><strong>مجموعة اختبار صغيرة.</strong> 69 صورة تعني أن أرقامنا تحمل
هامش خطأ واسعاً. لا تعامل الفرق بين 0.82 و 0.85 كفرق حقيقي.</li>
</ul>

<div class="note">
<p><strong>بُعد يتجاوز التقنية:</strong> نظام يقرّر فتح إشارة مرور بناءً
على كشف بصري يمسّ سلامة الناس. الخطأ فيه ليس رقماً في جدول: تفويت إسعاف
قد يعني تأخيراً في حالة حرجة، وإنذار كاذب متكرر قد يدفع الناس إلى تجاهل
النظام كله. أي نظام حقيقي من هذا النوع يحتاج مراقبة بشرية، وآلية تجاوز
يدوي، واختباراً ميدانياً طويلاً قبل أن يُعتمد عليه.</p>
</div>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>12. تمرين صفي</h1>

<h3>المهمة 1: أثر عدد الدورات</h3>
<p>أعد التدريب المصغّر بـ <code>epochs=10</code> بدل 5. سجّل
<code>mAP@50</code> في الحالتين. هل تضاعفت النتيجة بمضاعفة الدورات؟ ولماذا
في رأيك؟</p>

<h3>المهمة 2: تجميد الطبقات</h3>
<p>أضف <code>freeze=10</code> إلى خلية التدريب المصغّر. قارن زمن التدريب
والنتيجة النهائية. متى يكون هذا التنازل مقبولاً؟</p>

<h3>المهمة 3: افحص توسيمك بعينك</h3>
<p>اختر عشر صور من <code>dataset/train/images</code> واعرض صناديقها. هل
تجد صندوقاً خاطئاً أو ناقصاً أو أوسع من جسمه؟ سجّل ما تجد.</p>

<h3>المهمة 4: صورك أنت</h3>
<p>ضع صورة فيها مركبة في مجلد <code>student_images/</code>، وشغّل عليها
النموذجين. اشرح الفرق بينهما بجملتين.</p>

</div>


In [ ]:
# TODO:
# 1. غيّر القيم التالية
# 2. أعد بناء قاعدة البيانات المصغّرة إن لزم
# 3. شغّل التدريب وسجّل النتيجة
# 4. قارنها بنتيجتك الأولى

STUDENT_EPOCHS = 10
STUDENT_FREEZE = None      # جرّب 10
STUDENT_IMGSZ = 320

# اكتب الحل هنا


In [ ]:
# المهمة 4: شغّل النموذجين على صورك أنت
student_images = sorted(Path("student_images").glob("*.jpg"))

if not student_images:
    print("ضع صورة أو أكثر في مجلد student_images/ ثم أعد تشغيل هذه الخلية.")
else:
    for image_path in student_images[:3]:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        for ax, (title, model) in zip(
            axes, [("Pretrained", pretrained_model),
                   ("Fine-tuned", finetuned_model)]
        ):
            result = model.predict(
                source=str(image_path), conf=0.35, device=DEVICE, verbose=False
            )[0]
            ax.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
            ax.set_title(title, fontsize=14)
            ax.axis("off")

        plt.suptitle(image_path.name, fontsize=15)
        plt.tight_layout()
        plt.show()


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>13. أسئلة مراجعة سريعة</h1>

<h3>سؤال 1</h3>
<p>لماذا لا يمكن لضبط <code>conf</code> أن يجعل النموذج الجاهز يكشف
سيارة إسعاف؟</p>

<h3>سؤال 2</h3>
<p>ما الفرق بين تسمية على مستوى الصورة وتسمية على مستوى الجسم؟ وأيهما
يلزم لتدريب كاشف؟</p>

<h3>سؤال 3</h3>
<p>في السطر <code>0 0.48 0.61 0.18 0.27</code>، ماذا يمثّل كل رقم؟ ولماذا
كل القيم أصغر من واحد؟</p>

<h3>سؤال 4</h3>
<p>لماذا يُعد وضع إطارات متتالية من فيديو واحد في <code>train</code> و
<code>valid</code> معاً خطأً فادحاً؟</p>

<h3>سؤال 5</h3>
<p>نموذج Precision له 0.95 و Recall 0.40. صف سلوكه بكلماتك. وهل يصلح
لنظام أولوية الإسعاف؟</p>

<h3>سؤال 6</h3>
<p>ما الفرق بين <code>mAP@50</code> و <code>mAP@50-95</code>؟ ولماذا
الثاني أقل دائماً؟</p>

<h3>سؤال 7</h3>
<p>خسارة التدريب تنخفض وخسارة التحقق ترتفع. ما التشخيص؟ واذكر علاجين.</p>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>14. الخلاصة</h1>

<p>ما فعلناه في هذا الدرس:</p>

<ul>
<li>أثبتنا أن النموذج الجاهز <strong>يفشل فشلاً واثقاً</strong>: كشف
سيارة الإسعاف بثقة تجاوزت 0.90 وسمّاها <code>truck</code>، لأن الفئة
غير موجودة في قاموسه أصلاً.</li>
<li>ميّزنا بين <strong>فجوة الفئات</strong> التي لا يصلحها إلا التدريب،
و<strong>اختلاف المجال</strong> الذي قد تخفّفه العتبات.</li>
<li>فهمنا أن <strong>Fine-Tuning</strong> يعيد استخدام نحو 99% من الأوزان
ولا يستبدل إلا الرأس، ولهذا تكفيه مئات الصور لا ملايينها.</li>
<li>تعلّمنا صيغة توسيم YOLO وكتبناها بأيدينا، وبنينا بنية المجلدات وملف
<code>data.yaml</code>، وقسّمنا البيانات تقسيماً نزيهاً.</li>
<li>درّبنا فعلياً، وقرأنا الخسائر الثلاث ومنحنيات النتائج ومصفوفة
الالتباس، وشخّصنا التجهيز الزائد.</li>
<li>حصلنا على نموذج يعرف <code>ambulance</code> إلى جانب
<code>car</code> و <code>truck</code> و <code>bus</code>.</li>
</ul>

<h2>في الأسبوع القادم</h2>

<p>نموذجنا الآن يجيب عن سؤالَي «ما هذا؟» و«أين هو؟» في كل إطار على حدة.
لكنه <strong>ينسى كل شيء بين إطار وإطار</strong>: لا يعرف أن الإسعاف الذي
رآه في الإطار العاشر هو نفسه الذي رآه في الإطار التاسع.</p>

<p>ولهذا لا نستطيع بعد أن نجيب عن أسئلة بسيطة مثل: كم مركبة عبرت
التقاطع؟ وبأي سرعة يقترب هذا الإسعاف؟</p>

<p>حلّ ذلك هو <strong>التتبّع Object Tracking</strong>: أن نعطي كل جسم
رقماً ثابتاً يلازمه عبر الإطارات. وهذا موضوع الأسبوع الثامن.</p>

<hr />
<h2>مصادر للاستزادة</h2>

<ul>
<li><a href="https://docs.ultralytics.com/modes/train/">Ultralytics — Train</a></li>
<li><a href="https://docs.ultralytics.com/modes/val/">Ultralytics — Validate</a></li>
<li><a href="https://docs.ultralytics.com/datasets/detect/">Ultralytics — تنسيق قواعد بيانات الكشف</a></li>
<li><a href="https://docs.ultralytics.com/guides/yolo-performance-metrics/">Ultralytics — شرح مقاييس الأداء</a></li>
<li><a href="https://docs.ultralytics.com/guides/model-training-tips/">Ultralytics — نصائح للتدريب</a></li>
<li><a href="https://storage.googleapis.com/openimages/web/index.html">Open Images Dataset</a></li>
</ul>

</div>
